# 09c. Freeze the full-GAVD GaitParity training contract

This notebook turns the contracts in `nb_09a` and `nb_09b` into a concrete three-model
feasibility run over every available GAVD sequence:

1. a shared one-view **standard** encoder applied independently to both orbit members;
2. a **paired-unconstrained** encoder with branch-specific self- and cross-attention;
3. a **reflection-equivariant** encoder with shared self-attention and symmetric cross-attention.

This is not clinical evidence. GAVD lacks participant IDs, the encoder sees the local corpus, and all
health quantities come from the skeleton coordinates themselves. The allowed conclusion is only that
the full loop trains locally, remains non-collapsed, and satisfies its declared geometry.

### Safe startup

The notebook defaults to `GAIT_PARITY_MODE=smoke` and the CPU device. A full local run requires:

```bash
export GAIT_PARITY_MODE=real
export GAIT_PARITY_RUN_ID=gavd-full-v1
export GAIT_PARITY_PROFILE=cpu       # or gpu
export GAIT_PARITY_DEVICE=cpu        # CUDA is opt-in
export GAIT_PARITY_MATCHING=exposure # rerun as compute for the second fairness view
```


In [1]:
from pathlib import Path
import json, math, os, sys

def find_notebook_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "gait_parity_jepa.py").exists() and (candidate / "nb_09a_equivariant_encoder_contract.ipynb").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from experiments/sjepa/gavd6 or set the working directory there")

PROJECT_DIR = find_notebook_root()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from gait_parity_jepa import *

MODE = os.getenv("GAIT_PARITY_MODE", "smoke").strip().lower()
if MODE not in {"smoke", "real"}:
    raise ValueError("GAIT_PARITY_MODE must be smoke or real")
PROFILE_NAME = "smoke" if MODE == "smoke" else os.getenv("GAIT_PARITY_PROFILE", "cpu").strip().lower()
if PROFILE_NAME not in PROFILES or (MODE == "real" and PROFILE_NAME == "smoke"):
    raise ValueError("Real runs require GAIT_PARITY_PROFILE=cpu or gpu")
CONFIG = PROFILES[PROFILE_NAME]
MATCHING_REGIME = os.getenv("GAIT_PARITY_MATCHING", "exposure").strip().lower()
if MATCHING_REGIME not in {"exposure", "compute"}:
    raise ValueError("GAIT_PARITY_MATCHING must be exposure or compute")
RUN_ID = os.getenv("GAIT_PARITY_RUN_ID", "smoke")
if MODE == "real" and RUN_ID == "smoke":
    raise ValueError("Set a versioned GAIT_PARITY_RUN_ID for a real run")
SEEDS = [int(value) for value in os.getenv(
    "GAIT_PARITY_SEEDS", "7" if PROFILE_NAME in {"cpu", "smoke"} else "7,19,31"
).split(",")]
OUT_DIR = PROJECT_DIR / "work" / "artifacts" / "gait_parity" / MODE / RUN_ID / MATCHING_REGIME
OUT_DIR.mkdir(parents=True, exist_ok=True)

if MODE == "smoke":
    RECORDS = synthetic_records(frames=48)
    POSE_DIR = None
else:
    POSE_DIR = resolve_pose_dir(PROJECT_DIR)
    RECORDS = load_gavd_records(POSE_DIR)
WINDOWS, VALID_PATCH, WINDOW_TABLE = build_windows(RECORDS, CONFIG)
MANIFEST = cohort_manifest(RECORDS, WINDOW_TABLE, CONFIG, MODE)

print("scope            :", MANIFEST["scope"])
print("mode/profile     :", MODE, "/", PROFILE_NAME)
print("matching regime  :", MATCHING_REGIME)
print("run ID           :", RUN_ID)
print("device default   :", os.getenv("GAIT_PARITY_DEVICE", "cpu"))
print("records/windows  :", len(RECORDS), "/", len(WINDOWS))
print("source videos    :", MANIFEST["source_video_count"])
print("output           :", OUT_DIR)


scope            : local GAVD feasibility; transductive representation/geometry evidence only
mode/profile     : smoke / smoke
matching regime  : exposure
run ID           : smoke
device default   : cpu
records/windows  : 12 / 24
source videos    : 6
output           : /Users/theodoremui/dev/alexpose/experiments/sjepa/gavd6/work/artifacts/gait_parity/smoke/smoke/exposure


## 1. CPU and CUDA profiles

The CPU profile is intentionally modest but uses all 96 cached GAVD sequences. The GPU profile restores
the 96-wide, four-layer encoder scale, denser temporal windows, more epochs, more seeds, and mixed
precision. Changing hardware does not change the objective.


In [2]:
profile_table = pd.DataFrame([asdict(PROFILES[name]) for name in ["cpu", "gpu", "smoke"]]).set_index("profile")
display(profile_table)


,frames,stride,segment_length,embed_dim,encoder_depth,predictor_depth,heads,batch_size,epochs,learning_rate,weight_decay,mask_fraction,ema_start,vicreg_weight,odd_vicreg_weight,max_yaw_degrees,amp
profile,,,,,,,,,,,,,,,,,
cpu,64,64,8,32,2,1,4,8,6,0.0003,0.05,0.5,0.996,0.05,1.0,8.0,False
gpu,64,32,4,96,4,2,8,32,100,0.0002,0.05,0.6,0.996,0.05,1.0,8.0,True
smoke,32,32,8,16,1,1,4,4,1,0.0008,0.01,0.4,0.990,0.10,1.0,4.0,False


## 2. Freeze anatomical reflection and the full-cohort window manifest

Reflection negates the lateral coordinate and exchanges all 16 BlazePose left/right pairs. Windows are
formed only after short-gap interpolation, pelvis centering, and robust body scaling. Every eligible
sequence is retained. Conditions are recorded for provenance but never enter the objective.


In [3]:
mirror_error = float((anatomical_mirror(anatomical_mirror(WINDOWS[:8])) - WINDOWS[:8]).abs().max())
assert mirror_error == 0.0
assert MANIFEST["record_count"] == (12 if MODE == "smoke" else 96)
display(WINDOW_TABLE.groupby("condition").agg(sequences=("sequence_id", "nunique"), windows=("window_id", "size")))
print("mirror involution max abs:", mirror_error)
print("cohort SHA256:", MANIFEST["cohort_sha256"])
print("window SHA256:", MANIFEST["window_sha256"])


,sequences,windows
condition,,
illustrative,12,24


mirror involution max abs: 0.0
cohort SHA256: 28a0a023361eebd8fed3a25da726a19869408df939057a355379c6763862af9f
window SHA256: aac069a4a3e167c9b5409a60514a585c3bfe84e16731e0642c4e635d67b6b473


## 3. Freeze the shared objective and both matching views

All variants optimize the same centered-and-sharpened masked-token JEPA loss. The EMA teacher is updated
with the same schedule. The same two orbit augmentations enter parity-resolved VICReg: invariance,
variance, and covariance are evaluated separately on the even and odd channels. This is deliberately
stronger than total-feature VICReg because `nb_09a` demonstrated that a perfectly equivariant odd
channel can collapse to zero.

`exposure` gives every variant the same optimizer updates, orbit windows, masks, and branch forwards.
`compute` holds a frozen analytic token-parameter budget approximately constant, so architectures with
different per-step costs receive different update counts. The proxy is not called FLOPs; measured wall
time, peak CUDA memory, parameter counts, and actual exposures remain separate manifest fields.


In [4]:
models = {variant: build_model(CONFIG, variant, SEEDS[0]) for variant in VARIANTS}
updates = planned_updates(models, CONFIG, len(WINDOWS), MATCHING_REGIME)
architecture_rows = []
for variant, model in models.items():
    architecture_rows.append({
        "variant": variant,
        "trainable_parameters": parameter_count(model),
        "encoder_parameters": parameter_count(model.encoder),
        "compute_proxy_per_step": compute_proxy_per_step(model, CONFIG),
        "planned_updates": updates[variant],
        "planned_orbit_exposures": updates[variant] * CONFIG.batch_size,
    })
architecture_table = pd.DataFrame(architecture_rows).set_index("variant")
display(architecture_table)


,trainable_parameters,encoder_parameters,compute_proxy_per_step,planned_updates,planned_orbit_exposures
variant,,,,,
standard,8224,4304,18180096,6,24
paired_unconstrained,13744,9824,41496576,6,24
reflection_equivariant,9344,5424,22910976,6,24


## 4. Persist the immutable input to training

These gates are engineering feasibility thresholds, chosen without clinical outcomes. Passing them is
not evidence that a representation is useful for force prediction or unseen participants.


In [5]:
HEALTH_GATES = {
    "minimum_feature_variance": 1e-5,
    "minimum_effective_rank": 1.05,
    "maximum_mean_pairwise_cosine": 0.995,
    "minimum_odd_to_even_energy_ratio": 1e-5,
    "maximum_odd_to_even_energy_ratio": 1e5,
    "equivariance_float32_atol": 5e-5,
    "unconstrained_control_minimum_residual": 1e-6,
}
contract = {
    "notebook": "nb_09c_gavd_matched_jepa_contract",
    "scope": MANIFEST["scope"],
    "run_id": RUN_ID,
    "matching_regime": MATCHING_REGIME,
    "seeds": SEEDS,
    "pose_dir": str(POSE_DIR) if POSE_DIR else None,
    "train_config": asdict(CONFIG),
    "cohort_manifest": MANIFEST,
    "architectures": architecture_table.reset_index().to_dict(orient="records"),
    "objective": {
        "jepa": "centered_sharpened_latent_cross_entropy",
        "anti_collapse": "even_and_odd_orbit_VICReg",
        "condition_labels_used": False,
    },
    "health_gates": HEALTH_GATES,
}
WINDOW_TABLE.to_csv(OUT_DIR / "window_manifest.csv", index=False)
write_json(OUT_DIR / "training_contract.json", contract)
print("Wrote", OUT_DIR / "training_contract.json")


Wrote /Users/theodoremui/dev/alexpose/experiments/sjepa/gavd6/work/artifacts/gait_parity/smoke/smoke/exposure/training_contract.json
